# Shots Bottle Grab Tuning

Standalone arm-and-CAP2-base tuning notebook for small plastic vitamin-shot bottles. It does not use the camera, DepthNet, or object detection.

The sequence uses separate **CAP1**, **Push**, and **Grab** states. CAP1 opens and lowers the claw. Push applies a guarded forward base movement and the stored `cap2_*` arm pose without touching servo 4. Grab only closes servo 4. Every state has its own pause-after-completion parameter.

In [ ]:
import json
import time
import threading
from pathlib import Path

import ipywidgets as widgets
from IPython.display import display

# Start safe: True logs commands without moving the real hardware.
DRY_RUN_ARM = True
DRY_RUN_BASE = True
PARAM_PATH = Path('shots_bottle_grab_tuning_params.json')

ttl_servo = None
robot = None
base_stop_lock = threading.Lock()
if not DRY_RUN_ARM:
    from SCSCtrl import TTLServo
    ttl_servo = TTLServo

DEFAULT_PARAMS = {
    'safe_s1': 0, 'safe_s2': 0, 'safe_s3': 0, 'safe_s4': 0, 'safe_s5': 0,
    'safe_order': [5, 4, 3, 2, 1],
    'ready_s1': 0, 'ready_s2': 18, 'ready_s3': -12, 'ready_s4': 0, 'ready_s5': 0,
    'ready_order': [5, 4, 3, 2, 1],
    'cap1_s1': 0, 'cap1_s2': 18, 'cap1_s3': -12, 'cap1_s5': 0,
    'cap1_order': [5, 4, 3, 2, 1],
    'cap2_s1': 0, 'cap2_s2': 26, 'cap2_s3': -22, 'cap2_s5': 0,
    'cap2_order': [5, 4, 3, 2, 1],
    'lift_s1': 0, 'lift_s2': 0, 'lift_s3': 28, 'lift_s5': 0,
    'lift_order': [5, 4, 3, 2, 1],
    'gripper_open': 0,
    'gripper_close_1': -45,
    'gripper_close_2': -55,
    'arm_speed': 80,
    'gripper_speed': 120,
    'settle_seconds': 0.35,
    'cap_pause_seconds': 0.50,
    'cap2_base_speed': 0.15,
    'cap2_base_max_seconds': 0.25,
    'safe_pause_time': 0.50,
    'ready_pause_time': 0.50,
    'cap1_pause_time': 0.50,
    'push_pause_time': 0.50,
    'grab_pause_time': 0.50,
    'lift_pause_time': 0.50,
}


def load_params():
    loaded = dict(DEFAULT_PARAMS)
    if PARAM_PATH.exists():
        saved = json.loads(PARAM_PATH.read_text())
        loaded.update(saved)
        legacy_pause = float(saved.get('cap_pause_seconds', DEFAULT_PARAMS['cap_pause_seconds']))
        for name in ['safe_pause_time', 'ready_pause_time', 'cap1_pause_time', 'push_pause_time', 'grab_pause_time', 'lift_pause_time']:
            if name not in saved:
                loaded[name] = legacy_pause
    return loaded


params = load_params()
print('DRY_RUN_ARM =', DRY_RUN_ARM)
print('DRY_RUN_BASE =', DRY_RUN_BASE)
print('params loaded from', PARAM_PATH if PARAM_PATH.exists() else 'defaults')

In [ ]:
def angle_slider(name, min_value=-180, max_value=180):
    return widgets.IntSlider(
        value=int(params[name]), min=min_value, max=max_value, step=1,
        description=name, continuous_update=False, style={'description_width': '125px'},
        layout=widgets.Layout(width='520px')
    )


def speed_slider(name):
    return widgets.IntSlider(
        value=int(params[name]), min=20, max=300, step=5,
        description=name, continuous_update=False, style={'description_width': '125px'},
        layout=widgets.Layout(width='520px')
    )


def float_slider(name):
    return widgets.FloatSlider(
        value=float(params[name]), min=0.05, max=1.5, step=0.05,
        description=name, continuous_update=False, style={'description_width': '125px'},
        layout=widgets.Layout(width='520px')
    )


def base_speed_slider(name):
    return widgets.FloatSlider(
        value=float(params[name]), min=0.0, max=0.5, step=0.01,
        description=name, continuous_update=False, readout_format='.2f',
        style={'description_width': '175px'}, layout=widgets.Layout(width='560px')
    )


def base_time_slider(name):
    return widgets.FloatSlider(
        value=float(params[name]), min=0.05, max=2.0, step=0.05,
        description=name, continuous_update=False, readout_format='.2f',
        style={'description_width': '175px'}, layout=widgets.Layout(width='560px')
    )


def pause_slider(name):
    return widgets.FloatSlider(
        value=float(params[name]), min=0.0, max=15.0, step=0.05,
        description=name, continuous_update=False, readout_format='.2f',
        style={'description_width': '175px'}, layout=widgets.Layout(width='560px')
    )


def order_text(name):
    value = ','.join(str(item) for item in params[name])
    return widgets.Text(
        value=value, description=name, placeholder='5,4,3,2,1',
        style={'description_width': '125px'}, layout=widgets.Layout(width='520px')
    )


ANGLE_NAMES = [
    'safe_s1', 'safe_s2', 'safe_s3', 'safe_s4', 'safe_s5',
    'ready_s1', 'ready_s2', 'ready_s3', 'ready_s4', 'ready_s5',
    'cap1_s1', 'cap1_s2', 'cap1_s3', 'cap1_s5',
    'cap2_s1', 'cap2_s2', 'cap2_s3', 'cap2_s5',
    'lift_s1', 'lift_s2', 'lift_s3', 'lift_s5',
    'gripper_open', 'gripper_close_1', 'gripper_close_2',
]
sliders = {name: angle_slider(name) for name in ANGLE_NAMES}
sliders['arm_speed'] = speed_slider('arm_speed')
sliders['gripper_speed'] = speed_slider('gripper_speed')
sliders['settle_seconds'] = float_slider('settle_seconds')
sliders['cap_pause_seconds'] = float_slider('cap_pause_seconds')
sliders['cap2_base_speed'] = base_speed_slider('cap2_base_speed')
sliders['cap2_base_max_seconds'] = base_time_slider('cap2_base_max_seconds')
for name in ['safe_pause_time', 'ready_pause_time', 'cap1_pause_time', 'push_pause_time', 'grab_pause_time', 'lift_pause_time']:
    sliders[name] = pause_slider(name)
ORDER_NAMES = ['safe_order', 'ready_order', 'cap1_order', 'cap2_order', 'lift_order']
order_widgets = {name: order_text(name) for name in ORDER_NAMES}
sliders['cap_pause_seconds'].description = 'gripper_step_pause'
sliders['cap2_base_speed'].description = 'push_base_speed'
sliders['cap2_base_max_seconds'].description = 'push_base_max_time'
order_widgets['cap2_order'].description = 'push_order'


def parse_servo_order(value, name):
    cleaned = str(value).strip().replace('(', '').replace(')', '').replace('[', '').replace(']', '')
    parts = [part.strip() for part in cleaned.split(',') if part.strip()]
    try:
        order = [int(part) for part in parts]
    except ValueError:
        raise ValueError('{} must contain comma-separated servo IDs'.format(name))
    if len(order) != 5 or sorted(order) != [1, 2, 3, 4, 5]:
        raise ValueError('{} must contain each servo ID 1-5 exactly once'.format(name))
    return order


def numeric_params():
    return {name: slider.value for name, slider in sliders.items()}


def state_order(prefix):
    name = prefix + '_order'
    return parse_servo_order(order_widgets[name].value, name)


def current_params():
    data = numeric_params()
    for name, widget in order_widgets.items():
        data[name] = parse_servo_order(widget.value, name)
    return data


def save_params(_=None):
    data = current_params()
    PARAM_PATH.write_text(json.dumps(data, indent=2) + '\n')
    print('[params] saved to', PARAM_PATH)


def ensure_robot():
    global robot
    if DRY_RUN_BASE:
        return None
    if robot is None:
        from jetbot import Robot
        robot = Robot()
        print('[base] Robot connected')
    return robot


def stop_base(reason='manual'):
    with base_stop_lock:
        if robot is not None:
            robot.stop()
    print('[base] stop reason={}'.format(reason))


def start_cap2_base_push():
    p = numeric_params()
    speed = float(p['cap2_base_speed'])
    max_seconds = float(p['cap2_base_max_seconds'])
    bot = ensure_robot()
    print('[base] CAP2 forward speed={:.2f} max_seconds={:.2f} dry_run={}'.format(
        speed, max_seconds, DRY_RUN_BASE
    ))
    if bot is None:
        return None
    bot.forward(speed)
    timer = threading.Timer(max_seconds, lambda: stop_base('CAP2 maximum time reached'))
    timer.daemon = True
    timer.start()
    return timer


def test_cap2_base_push():
    p = numeric_params()
    timer = start_cap2_base_push()
    if timer is None:
        print('[base] dry-run push test complete')
        return
    try:
        time.sleep(float(p['cap2_base_max_seconds']) + 0.10)
    finally:
        timer.cancel()
        stop_base('CAP2 push test cleanup')


def move_servo(servo_id, angle, speed, label=''):
    print('[arm] servo={} angle={} speed={} {}'.format(servo_id, angle, speed, label))
    if ttl_servo is not None:
        ttl_servo.servoAngleCtrl(int(servo_id), int(angle), 1, int(speed))
    time.sleep(float(sliders['settle_seconds'].value))


def pause_after(state_name):
    key = state_name + '_pause_time'
    seconds = float(sliders[key].value)
    print('[pause] state={} seconds={:.2f}'.format(state_name, seconds))
    if seconds > 0:
        time.sleep(seconds)


def apply_ordered_state(name, prefix, servo4_action=None):
    p = numeric_params()
    order = state_order(prefix)
    print('[arm] state={} order={}'.format(name, tuple(order)))
    for servo_id in order:
        key = '{}_s{}'.format(prefix, servo_id)
        if servo_id == 4 and servo4_action is not None:
            servo4_action()
        elif key in p:
            move_servo(servo_id, p[key], p['arm_speed'], name)
        else:
            print('[arm] servo={} skipped in {}'.format(servo_id, name))


def safe_home():
    apply_ordered_state('safe_home', 'safe')
    pause_after('safe')


def ready_state():
    apply_ordered_state('ready_state', 'ready')
    pause_after('ready')


def open_gripper():
    p = numeric_params()
    move_servo(4, p['gripper_open'], p['gripper_speed'], 'open_gripper')


def close_gripper():
    p = numeric_params()
    move_servo(4, p['gripper_close_1'], p['gripper_speed'], 'close_1')
    time.sleep(float(p['cap_pause_seconds']))
    move_servo(4, p['gripper_close_2'], p['gripper_speed'], 'close_2')


def cap1_state():
    print('[state] CAP1: ordered open-and-lower state')
    apply_ordered_state('cap1_down', 'cap1', servo4_action=open_gripper)
    pause_after('cap1')


def push_state():
    print('[state] Push: guarded base movement plus cap2 arm pose; servo 4 unchanged')
    timer = start_cap2_base_push()
    try:
        apply_ordered_state('push_state', 'cap2')
    finally:
        if timer is not None:
            timer.cancel()
        stop_base('Push state cleanup')
    pause_after('push')


def grab_state():
    print('[state] Grab: close servo 4 only')
    stop_base('Grab requires stationary base')
    close_gripper()
    pause_after('grab')


def lift_state():
    print('[state] Lift: ordered arm movement; servo 4 remains unchanged')
    apply_ordered_state('lift_state', 'lift')
    pause_after('lift')


def run_grab_sequence(_=None):
    print('[flow] shots bottle grab sequence start')
    try:
        safe_home()
        ready_state()
        cap1_state()
        push_state()
        grab_state()
        lift_state()
        print('[flow] shots bottle grab sequence done')
    finally:
        print('[flow] final base stop and safe_home')
        stop_base('full sequence cleanup')
        safe_home()


save_button = widgets.Button(description='Save Params', button_style='info')
home_button = widgets.Button(description='Safe Home', button_style='warning')
ready_button = widgets.Button(description='Ready State', button_style='success')
cap1_button = widgets.Button(description='CAP1 Down')
push_button = widgets.Button(description='Push State')
grab_button = widgets.Button(description='Grab State')
base_test_button = widgets.Button(description='Test CAP2 Base Push')
base_stop_button = widgets.Button(description='Stop Base', button_style='danger')
lift_button = widgets.Button(description='Lift State')
open_button = widgets.Button(description='Open Gripper')
close_button = widgets.Button(description='Close Gripper')
run_button = widgets.Button(description='Run Full Sequence', button_style='danger')

save_button.on_click(save_params)
home_button.on_click(lambda _: safe_home())
ready_button.on_click(lambda _: ready_state())
cap1_button.on_click(lambda _: cap1_state())
push_button.on_click(lambda _: push_state())
grab_button.on_click(lambda _: grab_state())
base_test_button.on_click(lambda _: test_cap2_base_push())
base_stop_button.on_click(lambda _: stop_base('UI stop button'))
lift_button.on_click(lambda _: lift_state())
open_button.on_click(lambda _: open_gripper())
close_button.on_click(lambda _: close_gripper())
run_button.on_click(run_grab_sequence)

print('shots bottle tuning UI objects ready')

In [ ]:
button_row_1 = widgets.HBox([save_button, home_button, ready_button, cap1_button, push_button, grab_button])
button_row_2 = widgets.HBox([base_test_button, base_stop_button, lift_button])
button_row_3 = widgets.HBox([open_button, close_button, run_button])

ui = widgets.VBox([
    widgets.HTML('<b>Actions</b>'),
    button_row_1,
    button_row_2,
    button_row_3,
    widgets.HTML('<b>Safe Home</b>'),
    order_widgets['safe_order'],
    sliders['safe_pause_time'],
    sliders['safe_s1'], sliders['safe_s2'], sliders['safe_s3'], sliders['safe_s4'], sliders['safe_s5'],
    widgets.HTML('<b>Ready State</b>'),
    order_widgets['ready_order'],
    sliders['ready_pause_time'],
    sliders['ready_s1'], sliders['ready_s2'], sliders['ready_s3'], sliders['ready_s4'], sliders['ready_s5'],
    widgets.HTML('<b>CAP1 - Open and Lower Claw</b>'),
    order_widgets['cap1_order'],
    sliders['cap1_pause_time'],
    sliders['cap1_s1'], sliders['cap1_s2'], sliders['cap1_s3'], sliders['cap1_s5'],
    widgets.HTML('<b>Push State - Base and Arm Movement Only</b>'),
    order_widgets['cap2_order'],
    sliders['push_pause_time'],
    sliders['cap2_s1'], sliders['cap2_s2'], sliders['cap2_s3'], sliders['cap2_s5'],
    sliders['cap2_base_speed'], sliders['cap2_base_max_seconds'],
    widgets.HTML('<b>Grab State - Servo 4 Only</b>'),
    sliders['grab_pause_time'],
    sliders['gripper_close_1'], sliders['gripper_close_2'],
    widgets.HTML('<b>Lift State - Servo 4 Unchanged</b>'),
    order_widgets['lift_order'],
    sliders['lift_pause_time'],
    sliders['lift_s1'], sliders['lift_s2'], sliders['lift_s3'], sliders['lift_s5'],
    widgets.HTML('<b>CAP1 Gripper Open</b>'),
    sliders['gripper_open'],
    widgets.HTML('<b>Speed / Timing</b>'),
    sliders['arm_speed'], sliders['gripper_speed'], sliders['settle_seconds'], sliders['cap_pause_seconds'],
])

display(ui)

## Suggested tuning order

1. Keep `DRY_RUN_ARM = True` for the first run and inspect the command order.
2. Restart the kernel with `DRY_RUN_ARM = False`, then test **Safe Home** and **Ready State**.
3. With the gripper clear of the bottle, tune **CAP1 Down** until the open claw is at the correct height.
4. Keep `DRY_RUN_BASE = True` initially, then use **Test CAP2 Base Push** to tune the chassis speed and maximum movement time.
5. Use **Push State** to tune the combined chassis and arm movement. This state skips servo 4.
6. Use **Grab State** to tune the two servo-4 close values without moving the chassis or other arm servos.
7. Tune **Lift State** while preserving the closed servo-4 position. Lift does not send another servo-4 command.
8. Each movement-state order must contain servo IDs 1-5 exactly once. Servo 4 opens in CAP1 and is skipped in Push and Lift.
9. Save the parameters before running the full sequence. The full sequence stops the base and returns to Safe Home in `finally`.